# Gaussian parameter stopping

Separate stopping budgets let a Gaussian location-scale fit choose how many updates to spend on its mean and on its standard deviation. This study asks whether separate budgets can bring interval width within 1% of LightGBM CQR at nominal 90% coverage on two-predictor processes (processes in which two input columns affect the response), using the public cross-validation and update methods. The [protocol](gaussian_two_column/PROTOCOL.md) fixes the workflows, comparison and acceptance gate (the pass rule) before measurement.

Each of 50 independent confirmation draws has 1,000 analysis observations, a common 500/500 fitting/calibration partition and 2,000 independent evaluation observations. Predictors are independent Uniform(−1,1), with mean $\sin(2\pi x_1)+0.5x_2$. Gaussian standard deviation is 1 (constant), $0.3+1.5|x_1|$ (absolute), $0.3+1.5\exp(-8x_1^2)$ (bell) or $\exp(0.7x_1)$ (monotone); a constant-variance heavy-tail control uses Student-t errors with three degrees of freedom, divided by $\sqrt{3}$. Every method sees the same rows. Calibration and evaluation responses select neither the specification nor its stopping budget.

The target is at most 1% paired real-width excess over prespecified LightGBM CQR at nominal coverage 0.90, with eligibility window [0.88,0.92]. Point estimates and one-sided 95% upper bounds are distinct. Both data-only and supplied separate-budget workflows must pass on both bell and monotone scale, and their intervals must be finite and coverage eligible on every process. Ten development draws remain separate from confirmation.

:::{note} Recorded before the response-units change
The recorded mboostrs GaussianLSS and CQR results on this page are recorded results. They were recorded with mboostrs 0.2.3 or earlier, when distributional (LSS) and `Quantile` fits boosted the response in the units it was given. Since then these fits divide the response by its standard deviation, boost the rescaled response and map the fitted model back (`response_scale="auto"`), so a step has the same size in any units; `Quantile` also starts at the `tau`-quantile of the response, and takes steps of 0.3 when no control is given. Refitting on the current version gives different numbers. `response_scale=None` with the settings the page states reproduces the recorded fits; [Response units](../implementation/parity_and_bugs.ipynb#response-units) describes the change and its measured effect.
:::

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from benches.evidence_float_codec import decode_record

directory = root / "docs/book/benchmarks/gaussian_two_column"
summary = decode_record(json.loads((directory / "confirmation_summary.json").read_text()))
rows = summary["rows"]
gate = json.loads((directory / "confirmation_gate.json").read_text())
labels = {"discovery": "Ordinary discovery", "discovery_separate": "Discovery / separate budgets",
          "supplied_fixed": "Supplied / 100 updates", "supplied_diagonal": "Supplied / diagonal tuning",
          "supplied_separate": "Supplied / separate budgets", "mboost_cqr": "Supplied / CQR 1600"}

In [ ]:
def comparison_table(records, phase):
    leaders = {}
    for panel in {r["panel"] for r in records}:
        eligible = [r for r in records if r["panel"] == panel and r.get("coverage_eligible") and not r["failures"]]
        for key in ("width", "true_mean_mse", "interval_score"):
            leaders[panel, key] = min((r[key] for r in eligible), default=None)

    def value(row, key):
        if key not in row:
            return "unavailable"
        text = f"{row[key]:.3f}"
        if key == "width" and row.get(key + "_se") is not None:
            text += f" ± {row[key + '_se']:.3f}"
        if row.get("coverage_eligible") and row[key] == leaders.get((row["panel"], key)):
            text = f"**{text}**"
        return text

    frame = pl.DataFrame([{"Process": r["panel"], "Method": labels.get(r["method"], r["method"].replace("_", " ")),
        "Width": value(r,"width"), "Coverage": value(r,"coverage"), "Mean MSE": value(r,"true_mean_mse"),
        "Interval score": value(r,"interval_score"), "Seconds": value(r,"elapsed"),
        "Empty sets": r.get("empty_sets", 0), "Failures": r["failures"]} for r in records])
    # Only complete, coverage-eligible methods can be marked as descriptive leaders.
    return (GT(frame, rowname_col="Method", groupname_col="Process").tab_stubhead("Workflow")
        .tab_header(title=f"Every workflow: {phase} Gaussian efficiency",
                    subtitle="Smaller width, latent-mean MSE and score are better; coverage target 0.90")
        .tab_spanner(label="Prediction sets", columns=["Width", "Coverage", "Interval score", "Empty sets"])
        .fmt_markdown(columns=["Width", "Mean MSE", "Interval score"])
        .tab_source_note(md("Width is response-unit real length, shown as mean ± Monte Carlo SE across independent draws. "
          "Mean MSE uses the generating conditional mean, in squared response units. CQR points are pre-calibration "
          "band midpoints. Bold marks minima among complete coverage-eligible rows, not a new method selection. "
          "Empty sets count crossed comparator endpoints and are retained with zero width and zero containment. "
          "The interval-score expression grades reported endpoint pairs; on crossed pairs it depends on crossing size "
          "as well as empty-set membership. Seconds are descriptive full-workflow costs including tuning, with shared-fit "
          "preparation charged to each applicable workflow. Memory was not measured. Source: phase summary file (development_summary.json or confirmation_summary.json).")))

In [ ]:
comparison_table(rows, "confirmation (50 draws per process)")

## Confirmation targets

The prespecified workflows use either data-only discovery followed by budget refinement, or a supplied mean/scale formula followed by the same refinement. The 100-update supplied fit and diagonal tuning remain controls. This target evaluates model selection on the fitting rows only (proper training), so calibration and evaluation rows play no part in choosing the model; it does not require an untuned preset to be optimal.

In [ ]:
targets = [r for r in rows if r["panel"] in ("bell", "monotone")
           and r["method"] in ("discovery_separate", "supplied_separate")]
frame = pl.DataFrame([{"Process": r["panel"], "Workflow": labels[r["method"]], "Coverage": r["coverage"],
    "Excess %": r["width_excess"], "MC SE": r["width_excess_se"], "Upper 95%": r["width_upper95"],
    "Upper target": r["upper_target"]} for r in targets])
(GT(frame, rowname_col="Workflow", groupname_col="Process").tab_stubhead("Information track")
 .tab_header(title="Frozen one-percent width targets", subtitle="Paired percent excess over LightGBM CQR; smaller is better")
 .fmt_number(columns=["Coverage", "Excess %", "MC SE", "Upper 95%"], decimals=3)
 .tab_source_note("Each draw contributes 100 × (workflow width / reference width − 1). MC SE is in percentage points. "
                  "Upper 95% uses Student-t on 49 degrees of freedom. Eligibility requires both methods' mean coverage "
                  "in [0.88,0.92]. The target uses the upper bound ≤ 1%, not a favorable point estimate alone."))

In [ ]:
bell_discovery = next(r for r in targets if r["panel"] == "bell" and r["method"] == "discovery_separate")
monotone_discovery = next(r for r in targets if r["panel"] == "monotone" and r["method"] == "discovery_separate")
fixed_bell = next(r for r in rows if r["panel"] == "bell" and r["method"] == "supplied_fixed")
assert gate["workflow_confirmation_pass"]
stronger = [{"Process": r["panel"], "Workflow": labels[r["method"]],
    "Largest adjusted upper %": max(c["upper_adjusted95"] for c in r["comparisons"] if c["eligible"]),
    "Eligible controls": sum(c["eligible"] for c in r["comparisons"])} for r in targets]
(GT(pl.DataFrame(stronger), rowname_col="Workflow", groupname_col="Process").tab_stubhead("Information track")
 .tab_header(title="All fixed competitor comparisons", subtitle="Largest Bonferroni-adjusted upper width excess; target ≤ 1%")
 .fmt_number(columns=["Largest adjusted upper %"], decimals=3)
 .tab_source_note("The one-sided bound uses the five-comparator critical value t(0.99,49). "
                  "Every fixed control remains eligible on these target processes; no confirmation winner is selected."))

Both prespecified information tracks meet the upper-bound target on bell and monotone scale. Data-only upper limits are {eval}`f"{bell_discovery['width_upper95']:.3f}%"` and {eval}`f"{monotone_discovery['width_upper95']:.3f}%"`. All five competitor comparisons also meet the separately adjusted criterion. The 100-update supplied bell preset remains at {eval}`f"{fixed_bell['width_excess']:.2f}%"` mean excess, with upper limit {eval}`f"{fixed_bell['width_upper95']:.2f}%"`.

The tables also report ordinary discovery and supplied diagonal tuning. Separate
mean/scale budgets meet the prespecified gate but do not uniformly improve on
diagonal tuning. This result concerns the stated smooth two-predictor processes and current proper-training workflow. The scale-attribution limitation for sharp mean functions also remains.

## Stopping-budget grid

The mean update counts are 25, 50, 100, 200 and 400; scale counts are 0, 25, 50, 100, 200 and 400. Zero scale updates retain the initial constant scale. The 30-point Cartesian grid is fixed in the protocol. Cross-validation uses five folds of fitting rows only. Each non-diagonal combination is fitted afresh in Rust because a cyclic location-scale path cannot be sliced into arbitrary component budgets. Exact ties retain the first grid position; no approximate tolerance is introduced.

Ordinary discovery uses the actual public ceilings of 400 for a mean-only model and 1,600 per parameter for an adopted LSS model. The refinement grid is capped at 400 per parameter. The [protocol](gaussian_two_column/PROTOCOL.md) records this distinction. Supplied diagonal tuning (separate tuning of the mean and scale with a single shared update count) starts at 100 and is capped at 400, at rate 0.1; its formulas are `mu: y ~ bbs(x1, df=4) + bols(x2)` and `sigma: y ~ bbs(x1, df=4)` on varying-scale processes, or `sigma: y ~ 1` on constant controls. Supplied CQR uses the mean formula with separate 0.05/0.95 quantile fits, 1,600 fixed updates and rate 0.1.

CatBoost split/CQR use 500 trees, depth 6 and rate 0.05. LightGBM split/CQR use 300 trees, 15 leaves and rate 0.05. sklearn CQR uses 300 trees, depth 3 and rate 0.05. Every fixed competitor uses the draw seed and one thread. Its settings are retained unchanged, rather than optimized against these test responses.

## Public workflow

The example repeats the first frozen bell confirmation input. It illustrates selection, budget tuning and prediction without adding a new confirmation observation. The simulated mean and scale define the grading process; neither is supplied to discovery.

In [ ]:
import mboostrs as mb
assert Path(mb.__file__).resolve().is_relative_to(root)

def simulate_bell(seed, n):
    rng = np.random.default_rng(seed)
    x1, x2 = rng.uniform(-1, 1, (2, n))
    scale = .3 + 1.5 * np.exp(-8*x1*x1)
    response = np.sin(2*np.pi*x1) + .5*x2 + scale*rng.normal(size=n)
    return pl.DataFrame({"x1": x1, "x2": x2, "y": response})

seed = 6305100
analysis = simulate_bell(seed, 1000)
evaluation = simulate_bell(seed+100000, 2000)
assignment = mb.cv_weights(analysis.height, type="kfold", B=2, seed=seed)[:, 0]
fitting = analysis.filter(pl.Series(assignment > 0))
calibration = analysis.filter(pl.Series(assignment == 0))

In [ ]:
selected = mb.discover(fitting, "y", seed=seed)
model = selected.model
choice = pl.DataFrame({"Field": ["Mean", "Scale", "Family"],
    "Value": [selected.formula, str(selected.spread_formula), selected.family]})
(GT(choice).tab_header(title="Data-only specification", subtitle="Selection uses the 500 fitting observations")
 .tab_source_note("The response's generating formula is not passed to this call."))

In [ ]:
grid = [(mu, sigma) for mu in (25, 50, 100, 200, 400) for sigma in (0, 25, 50, 100, 200, 400)]
if isinstance(model, mb.LSSModel):
    folds = mb.cv_weights(fitting.height, type="kfold", B=5, seed=seed)
    cv = model.cvrisk(folds=folds, grid=grid)
    budget = dict(zip(cv.param_names, (int(value) for value in cv.optimal_mstop), strict=True))
    refined = model.update(mstop=budget)
else:
    # A mean-only result is retained; a supplied scale formula is never substituted.
    budget, refined = {"mean": selected.mstop}, model
print(f"Selected update budgets: {budget}")

The calibration rows enter after specification and budget selection. `extrapolation="clip"` holds spline covariates at the nearest training boundary. The Gaussian location-scale fit calibrates its existing central distributional band; a mean-only result uses absolute residuals.

In [ ]:
band = mb.conformal_interval(refined, evaluation.drop("y"), calibration=calibration,
                             level=.9, seed=seed, extrapolation="clip")
example = pl.DataFrame({"Observed": evaluation["y"].head(5), "Mean": band.fitted_mean[:5],
                       "Lower": band.lower[:5], "Upper": band.upper[:5]})
(GT(example).tab_header(title="Held-out calibrated predictions", subtitle="Response units; first five evaluation rows")
 .fmt_number(columns=["Observed", "Mean", "Lower", "Upper"], decimals=3)
 .tab_source_note("The fitted distribution supplies its central band, with the legacy additive calibration correction. "
                  "Standardized-score development did not justify replacing that default."))

In [ ]:
# confirmation_summary.json keeps the recorded predictions of this confirmation draw
# (recorded before response_scale="auto" became the default for GaussianLSS).
recorded = summary["public_workflow_reference"]
assert (recorded["panel"], recorded["seed"], recorded["method"]) == ("bell", seed, "discovery_separate")
y_eval = evaluation["y"].to_numpy()
rec_lo = np.asarray(recorded["predictions"]["lower"])
rec_hi = np.asarray(recorded["predictions"]["upper"])
rec_pt = np.asarray(recorded["predictions"]["point"])
assert np.all(np.isfinite(band.lower)) and np.all(np.isfinite(band.upper))
live_cov = float(np.mean((y_eval >= band.lower) & (y_eval <= band.upper)))
rec_cov = float(np.mean((y_eval >= rec_lo) & (y_eval <= rec_hi)))
live_w = float(np.mean(band.upper - band.lower))
rec_w = float(np.mean(rec_hi - rec_lo))
print(f"Live response_scale=auto: coverage={live_cov:.4f}, mean width={live_w:.4f} "
      f"(recorded pre-scaling: coverage={rec_cov:.4f}, mean width={rec_w:.4f}).")

## Development and validation

Development seeds 6303100–6303109 precede independent confirmation seeds 6305100–6305149. All 550 development records and 2,750 confirmation records completed. The independent evaluator checks every archive hash, complete method/draw coverage, common partitions and inputs, and recomputes widths, coverage, interval scores and point errors from full stored predictions. Empty comparator sets remain in coverage and width summaries; failed methods would have unavailable full-method aggregates rather than averages over their successful draws.

The deeper audit regenerates every input and row assignment, replays actual native score/cutoff/endpoint calls, reconstructs competitor bands from their actual calibration/query predictions, and verifies the exact finite-grid minimum and its component budgets. Metadata keeps the source revision, loaded Python hashes, native build identity, package versions and thread settings. Conditional coverage uses the true-scale lower/upper thirds; in constant-scale controls ties place every row in both views. The complete records also retain warnings, risk matrices and workflow costs.

In [ ]:
development = decode_record(json.loads((directory / "development_summary.json").read_text()))["rows"]
comparison_table(development, "development (10 draws per process)")

## Reproduction

 Measurement uses the matching release native extension and repository-local Python. From the repository root:

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_gaussian_two_column.py \
  --phase confirmation --panel bell --output /tmp/gaussian-bell.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_gaussian_two_column.py --phase confirmation
PYTHONPATH=python:. .venv/bin/python benches/verify_gaussian_two_column.py
```

The runner refuses to overwrite evidence. The summary and audit commands validate the committed complete cohort; a reproduction on another revision is a separate run. This notebook's download contains the protocol, metadata, summary (`development_summary.json`, `confirmation_summary.json`) and validation files below. The per-draw archives are assets of the `benchmark-raw-archives-1` GitHub release; `docs/book/benchmarks/raw_archives.json` lists each file with its size, SHA-256 and the command that regenerates it. Whole-workflow elapsed costs are descriptive under concurrent panel jobs, rather than controlled implementation speed ratios.

**Limitation.** The recorded metadata omits the CatBoost and LightGBM version fields. The `ENVIRONMENT_ADDENDUM.json` file is a post-run installation inventory, and the measured versions are inferred from it, not captured at measurement time.

In [ ]:
artifacts = sorted(p.name for p in directory.iterdir() if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz")))
(GT(pl.DataFrame({"Artifact": artifacts})).tab_header(title="Downloaded files",
    subtitle="Protocols, metadata, summaries and independent validation")
 .tab_source_note("The per-draw XZ archives (strict JSON with reversible nonfinite locations, restored by the benchmark codec) are assets of the benchmark-raw-archives-1 GitHub release, listed in raw_archives.json; the metadata files above describe them."))